# 1 - Setup

In [ ]:
%%capture
!pip install -q "transformers==4.57.1" accelerate genomic-benchmarks

# 2 - Model & Tokenizer download

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species" #small model that can run on Colab on CPU only resources

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to("cpu").eval()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

esm_config.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species:
- esm_config.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_esm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species:
- modeling_esm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/224M [00:00<?, ?B/s]

In [ ]:
sequence = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"

In [ ]:
tokens = tokenizer.tokenize(sequence) # Inspect how the sequence is split into tokens

print("Original sequence:")
print(sequence)

print("\nTokens:")
print(tokens)

Original sequence:
ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC

Tokens:
['ACGTAG', 'CATCGG', 'ATCTAT', 'CTATCG', 'ACACTT', 'GGTTAT', 'CGATCT', 'ACGAGC', 'ATCTCG', 'T', 'T', 'A', 'G', 'C']


In [ ]:
inputs = tokenizer(
    sequence,
    return_tensors="pt"
)    # Convert the sequence into model-ready tensors

In [ ]:
import torch

with torch.inference_mode():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

# Take the embedding produced for each token by the final transformer layer
token_embeddings = outputs.hidden_states[-1]

print("Number of sequences:", token_embeddings.shape[0])
print("Number of tokens:", token_embeddings.shape[1])
print("Embedding size per token:", token_embeddings.shape[2])

Number of sequences: 1
Number of tokens: 15
Embedding size per token: 512


In [ ]:
token_index = 0 # select the first token

embedding = token_embeddings[0, token_index] # [0] selects the first (and only) sequence

print("Token:", tokens[token_index])

print("\nEmbedding shape:", embedding.shape) # number of dimensions in the token embedding

print("\nEmbedding values:")
print(embedding[:10])    # print first 10
print("...")
print(embedding[-10:])   # print last 10

Token: ACGTAG

Embedding shape: torch.Size([512])

Embedding values:
tensor([ 0.0253,  0.5766,  0.1433,  0.2284,  0.2655, -0.1052, -0.6204,  0.6659,
        -0.1597, -0.2938])
...
tensor([ 0.1969,  0.1234,  0.2801,  0.1974, -0.0708, -0.4174,  0.0267, -0.4804,
        -0.0430,  0.0638])


# 3 - Use a fine-tuned foundation model to make prediction

In [ ]:
import pandas as pd

URL = (
    "https://zenodo.org/records/15324459/files/human_enhancers_cohn_test.csv.gz"
)

test_df = pd.read_csv(URL)  # store the test dataset in a dataframe

In [ ]:
test_df.head(10)

,sequence,label
0,TCTTTTAATGAAGGGAAATATGGGATCTAAGTGAGGACACGGGCTT...,0
1,AATAGACTGAGTGGGGAAACTCAGCAAGGCCCATCTAGATTTTTCT...,1
2,CCTTTCAGGCGTTCTGATGATTTACCACAACATGGGTTGAAAGCCT...,0
3,CTAGCTGCACAGAAACTGCCACTTGGTGTGTAGGGGGTGTAGCGTA...,0
4,CTAAATGTTTATGTCTGGATGTGCAAGGAAGAAGCCGAGCAGTTCT...,0
5,ACATCTGGGGAAATGTCCTGGAAAAACCAAGGGAGCCAACATTCAT...,0
6,ACTCAAGAACAAATTCTATTTATTTATTATTGGAAAATGAAAAGCA...,1
7,TCAATAAAATTCCACCTATGCCCCCAAAGAATTCAAGTGTAATATT...,1
8,TAAATTAGTTGGTTTTGTTGCTTGGGCTTCAAAGAATGGCTTTATA...,0
9,CACAAACTTGCTTGGGCTAATGGAATGTGAGTGGAGATGACAGTCT...,1


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = (
    "tanoManzo/nucleotide-transformer-v2-50m-multi-species_ft_BioS11_1kbpHG19_DHSs_H3K27AC"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to("cuda").eval()  # load the fine-tuned classifier

In [ ]:
from tqdm.auto import tqdm
import torch

sequences = test_df["sequence"].tolist()

batch_size = 32

all_predictions = []
all_probabilities = []

for start in tqdm(range(0, len(sequences), batch_size)):
    batch_sequences = sequences[start:start + batch_size]

    # Tokenize and move inputs
    inputs = tokenizer(
        batch_sequences,
        return_tensors="pt"
    ).to("cuda")

    # Run inference
    with torch.inference_mode():
        outputs = classifier(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predictions = probabilities.argmax(dim=-1)

    all_predictions.extend(predictions.cpu().tolist())
    all_probabilities.extend(probabilities.cpu().tolist())

  0%|          | 0/218 [00:00<?, ?it/s]

In [ ]:
test_df["prediction"] = all_predictions

test_df["confidence"] = [
    max(probs)
    for probs in all_probabilities
]

test_df.head()

,sequence,label,prediction,confidence
0,TCTTTTAATGAAGGGAAATATGGGATCTAAGTGAGGACACGGGCTT...,0,0,0.983164
1,AATAGACTGAGTGGGGAAACTCAGCAAGGCCCATCTAGATTTTTCT...,1,1,0.584049
2,CCTTTCAGGCGTTCTGATGATTTACCACAACATGGGTTGAAAGCCT...,0,0,0.851616
3,CTAGCTGCACAGAAACTGCCACTTGGTGTGTAGGGGGTGTAGCGTA...,0,0,0.974053
4,CTAAATGTTTATGTCTGGATGTGCAAGGAAGAAGCCGAGCAGTTCT...,0,0,0.980830


In [ ]:
test_df["probability_class_1"] = [
    probs[1]
    for probs in all_probabilities
]

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(
    test_df["label"],
    test_df["prediction"]
)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(test_df["label"], test_df["prediction"]))

Accuracy: 0.6664
              precision    recall  f1-score   support

           0       0.62      0.88      0.72      3474
           1       0.79      0.46      0.58      3474

    accuracy                           0.67      6948
   macro avg       0.70      0.67      0.65      6948
weighted avg       0.70      0.67      0.65      6948

